# Fase 2 — Exploração

Este caderno é a versão **visual** do relatório `docs/relatorios/fase2.md`. O relatório
tem os números e fica no Git; aqui ficam os gráficos, para olhar e mexer.

Ele é gravado **sem as saídas**, de propósito: gráfico salvo dentro do `.ipynb` vira um
arquivo enorme e um diff ilegível. Rode as células (`Shift+Enter`) para ver as figuras.

**Antes de rodar:** `python scripts/preparar_dados.py`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from futebol.avaliacao import exploracao, metricas
from futebol.config import carregar_config
from futebol.dados import limpeza
from futebol.odds import mercado

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

cfg = carregar_config()
jogos = limpeza.carregar(cfg)

# Regra 13: toda análise diz de quais ligas ela fala.
grupo1 = jogos[jogos["grupo"] == "grupo1"]   # 22 ligas: backtest e CLV
grupo2 = jogos[jogos["grupo"] == "grupo2"]   # 16 países: treino e calibração

print(f"{len(jogos):,} jogos  |  Grupo 1: {len(grupo1):,}  |  Grupo 2: {len(grupo2):,}")

## 1. Os gols se parecem com uma Poisson?

A Fase 3 vai modelar gols como uma Poisson. Antes disso, vale olhar o quanto essa
hipótese se sustenta — e **onde** ela falha.

In [ ]:
distribuicao = exploracao.distribuicao_gols(jogos, maximo=8)

eixo = distribuicao.plot.bar(x="gols", y=["observado", "poisson"], width=0.8)
eixo.set_title("Gols por jogo: observado x Poisson de mesma média (38 competições)")
eixo.set_xlabel("gols no jogo")
eixo.set_ylabel("proporção dos jogos")
eixo.legend(["observado", "Poisson"])
plt.show()

distribuicao.style.format(
    {"observado": "{:.3%}", "poisson": "{:.3%}", "diferenca": "{:+.3%}"}
)

A Poisson acerta o formato geral e erra de um jeito específico: **jogos de 0 gol
acontecem mais do que ela prevê**. É esse desvio que o ajuste de Dixon-Coles corrige,
inflando a probabilidade dos placares baixos.

## 2. Quanto vale jogar em casa — e o que aconteceu em 2020

O gráfico mais interessante da fase. Em 2020 e 2021 os estádios ficaram vazios, e a
vantagem de mando caiu em quase todas as ligas do mundo **ao mesmo tempo**.

In [ ]:
por_temporada = exploracao.vantagem_mando(grupo1, ["temporada"]).sort_values("temporada")

figura, (esquerda, direita) = plt.subplots(1, 2, figsize=(13, 4.5))
esquerda.plot(por_temporada["temporada"], por_temporada["pontos_casa"], marker="o")
esquerda.set_title("Pontos por jogo do mandante (Grupo 1)")
direita.plot(
    por_temporada["temporada"], por_temporada["saldo_gols"], marker="o", color="tab:orange"
)
direita.set_title("Saldo de gols do mandante (Grupo 1)")

for eixo in (esquerda, direita):
    eixo.axvspan(-0.3, 1.3, color="tab:red", alpha=0.10)  # temporadas sem público
    eixo.tick_params(axis="x", rotation=45)

figura.suptitle("A queda do mando nas temporadas sem público (faixa vermelha)")
plt.tight_layout()
plt.show()

In [ ]:
# Liga a liga: quem mais sentiu a falta da torcida. Inclui os campeonatos de ano
# civil, que chamam o mesmo período de "2020" e "2021".
queda = exploracao.queda_do_mando_na_pandemia(jogos).sort_values("queda_pontos_casa")

eixo = queda.plot.barh(x="liga", y="queda_pontos_casa", legend=False, figsize=(9, 10))
eixo.set_title("Queda do mando sem público\nbarra positiva = o mandante perdeu pontos")
eixo.set_xlabel("pontos por jogo perdidos pelo mandante")
eixo.axvline(0, color="black", linewidth=0.8)
plt.show()

caiu = int((queda["queda_pontos_casa"] > 0).sum())
print(f"O mando caiu em {caiu} das {len(queda)} competições comparáveis.")

## 3. Quanto a casa cobra

O *overround* é a soma das probabilidades implícitas menos 1: quanto menor, melhor
para quem aposta.

⚠️ Margem baixa **não** quer dizer fácil de ganhar — nas grandes ligas ela é baixa
justamente porque o mercado é eficiente.

In [ ]:
ranking = exploracao.ranking_de_margem(jogos).sort_values("margem_media", ascending=False)
cores = ranking["grupo"].map({"grupo1": "tab:blue", "grupo2": "tab:gray"})
corte = cfg.secao("filtro_qualidade_mercado")["margem_maxima"]

eixo = ranking.plot.barh(
    x="liga", y="margem_media", color=list(cores), legend=False, figsize=(9, 10)
)
eixo.set_title(
    "Margem da casa no 1X2 de fechamento\n"
    "azul = Grupo 1 (pode apostar)  |  cinza = Grupo 2 (só treino)"
)
eixo.set_xlabel("overround médio")
eixo.axvline(corte, color="tab:red", linestyle="--")
eixo.text(corte, 0.5, " corte do filtro", color="tab:red")
plt.show()

In [ ]:
# A margem mudou ao longo dos anos? (Grupo 1, fechamento)
por_temporada, resumo = exploracao.evolucao_margem(grupo1)

pivo = por_temporada.pivot(index="temporada", columns="liga", values="margem_media")
eixo = pivo.plot(figsize=(12, 6), marker="o", alpha=0.75)
eixo.set_title("Margem por liga, temporada a temporada")
eixo.set_ylabel("overround médio")
eixo.legend(ncol=4, fontsize=8)
plt.show()

resumo.sort_values("variacao").style.format(
    {"primeira": "{:.2%}", "ultima": "{:.2%}", "variacao": "{:+.2%}"}
)

## 4. O mercado é bem calibrado?

Quando a odd diz 60%, acontece 60%? É a pergunta que define o tamanho do adversário.

In [ ]:
observado = mercado.resultado_observado(grupo1, "1x2")

figura, eixo = plt.subplots(figsize=(6.5, 6.5))
eixo.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1, label="calibração perfeita")

for metodo in mercado.METODOS:
    probabilidades = mercado.probabilidades_do_mercado(grupo1, "1x2", "fech", metodo)
    tabela = metricas.tabela_calibracao(probabilidades, observado)
    eixo.plot(
        [linha["previsto"] for linha in tabela],
        [linha["observado"] for linha in tabela],
        marker="o",
        label=f"{metodo} (ECE {metricas.ece(probabilidades, observado):.4f})",
    )

eixo.set_title("Calibração do mercado de fechamento (Grupo 1)")
eixo.set_xlabel("probabilidade que a odd dizia")
eixo.set_ylabel("frequência com que aconteceu")
eixo.legend()
plt.show()

In [ ]:
# O viés favorito-azarão, de perto: a diferença faixa a faixa.
probabilidades = mercado.probabilidades_do_mercado(grupo1, "1x2", "fech", "power")
tabela = pd.DataFrame(metricas.tabela_calibracao(probabilidades, observado))

eixo = tabela.plot.bar(x="faixa", y="diferenca", legend=False, color="tab:purple")
eixo.set_title("Aconteceu menos (−) ou mais (+) do que a odd dizia")
eixo.set_ylabel("diferença (proporção)")
eixo.axhline(0, color="black", linewidth=0.8)
plt.show()

tabela.style.format({"previsto": "{:.2%}", "observado": "{:.2%}", "diferenca": "{:+.2%}"})

Os **favoritos vencem mais** do que a odd dizia e os **azarões vencem menos**. É o
*viés favorito-azarão*, e ele sobrevive mesmo depois de tirar a margem pelo método que
melhor o corrige.

## 5. Mais de 2,5 gols, por liga

O mercado de Over/Under visto pelo lado do resultado.

In [ ]:
over = exploracao.frequencia_over25(jogos, ["grupo", "liga"]).sort_values("over25")
cores = over["grupo"].map({"grupo1": "tab:blue", "grupo2": "tab:gray"})

eixo = over.plot.barh(
    x="liga", y="over25", legend=False, figsize=(9, 10), color=list(cores)
)
eixo.set_title("Jogos com mais de 2,5 gols")
eixo.axvline(0.5, color="black", linewidth=0.8, linestyle="--")
plt.show()

## Para onde isso vai

O relatório completo, com todas as tabelas e as quatro respostas da fase, está em
[`docs/relatorios/fase2.md`](../docs/relatorios/fase2.md).

A Fase 3 recebe daqui: a meta de log loss a bater, o alerta sobre os placares de poucos
gols, o requisito de fator casa variável no tempo e a lista de ligas aprovadas.